In [1]:
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

from typing import Tuple, List
from tqdm.auto import tqdm, trange
from IPython import display

import librosa
import librosa.display
from IPython.display import Audio

In [2]:
random.seed(1337)
np.random.seed(1337)
torch.manual_seed(1337)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
!pip install soundata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.0/162.0 kB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 9.4 MB/s eta 0:00:00


In [4]:
import soundata
dataset = soundata.initialize('urbansound8k')

In [5]:
import soundata
import os
import pandas as pd

dataset = soundata.initialize("urbansound8k")
dataset.download()

DATASET_PATH = dataset.data_home
METADATA_PATH = os.path.join(DATASET_PATH, "metadata", "UrbanSound8K.csv")

print("DATASET_PATH:", DATASET_PATH)
print("METADATA_PATH:", METADATA_PATH)
print("CSV exists:", os.path.exists(METADATA_PATH))

metadata = pd.read_csv(METADATA_PATH)
print(len(metadata))
print(metadata["class"].value_counts().sort_index())

INFO: Downloading ['all', 'index']. Index is being stored in /usr/local/lib/python3.12/dist-packages/soundata/datasets/indexes, and the rest of files in /root/sound_datasets/urbansound8k
INFO: [all] downloading UrbanSound8K.tar.gz
5.61GB [05:59, 16.8MB/s]                                
INFO: [index] downloading urbansound8k_index_1.0.json
1.15MB [00:01, 1.01MB/s]                            

DATASET_PATH: /root/sound_datasets/urbansound8k
METADATA_PATH: /root/sound_datasets/urbansound8k/metadata/UrbanSound8K.csv
CSV exists: True
8732
class
air_conditioner     1000
car_horn             429
children_playing    1000
dog_bark            1000
drilling            1000
engine_idling       1000
gun_shot             374
jackhammer          1000
siren                929
street_music        1000
Name: count, dtype: int64


In [6]:
# Единый LabelEncoder

ALL_CLASSES = sorted(metadata["class"].unique())
global_label_encoder = LabelEncoder()
global_label_encoder.fit(ALL_CLASSES)

print(f"Классы ({len(global_label_encoder.classes_)}):")
print(list(global_label_encoder.classes_))

Классы (10):
[np.str_('air_conditioner'), np.str_('car_horn'), np.str_('children_playing'), np.str_('dog_bark'), np.str_('drilling'), np.str_('engine_idling'), np.str_('gun_shot'), np.str_('jackhammer'), np.str_('siren'), np.str_('street_music')]


In [10]:
import os
import uuid
import numpy as np
import torch
from torch.utils.data import Dataset
import librosa


class UrbanSoundDataset(Dataset):
    def __init__(
        self,
        metadata,
        dataset_path,
        label_encoder,
        target_length=4,
        sr=22050,
        n_mels=128,
        augment=False,
        cache_dir=None,
    ):
        self.metadata = metadata.reset_index(drop=True).copy()
        self.dataset_path = dataset_path
        self.label_encoder = label_encoder
        self.target_length = target_length
        self.sr = sr
        self.n_mels = n_mels
        self.augment = augment
        self.target_samples = int(target_length * sr)

        if cache_dir is None:
            cache_dir = os.path.join(dataset_path, "mel_cache")
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)

        self.labels = self.label_encoder.transform(self.metadata["class"])
        self.num_classes = len(self.label_encoder.classes_)

        self.expected_frames = 1 + self.target_samples // 512
        self.expected_shape = (self.n_mels, self.expected_frames)

    def __len__(self):
        return len(self.metadata)

    def load_audio(self, file_path):
        y, _ = librosa.load(file_path, sr=self.sr, mono=True)

        if len(y) > self.target_samples:
            y = y[:self.target_samples]
        elif len(y) < self.target_samples:
            y = np.pad(y, (0, self.target_samples - len(y)), mode="constant")

        return y.astype(np.float32)

    def extract_mel_spectrogram(self, y):
        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=self.sr,
            n_mels=self.n_mels,
            n_fft=2048,
            hop_length=512,
            fmax=8000
        )
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)

        mean = mel_db.mean()
        std = mel_db.std() + 1e-6
        mel_db = (mel_db - mean) / std

        return mel_db.astype(np.float32)

    def augment_audio(self, y):
        if np.random.rand() < 0.5:
            shift = np.random.randint(-self.sr // 2, self.sr // 2)
            y = np.roll(y, shift)

        if np.random.rand() < 0.3:
            noise = np.random.randn(len(y)).astype(np.float32) * 0.005
            y = y + noise

        return np.clip(y, -1.0, 1.0).astype(np.float32)

    def spec_augment(self, mel_db, max_time_masks=2, max_freq_masks=2, max_time_width=20, max_freq_width=10):
        mel = mel_db.copy()
        num_mels, num_frames = mel.shape

        for _ in range(max_freq_masks):
            f = np.random.randint(0, max_freq_width + 1)
            if f == 0:
                continue
            f0 = np.random.randint(0, max(1, num_mels - f))
            mel[f0:f0+f, :] = 0

        for _ in range(max_time_masks):
            t = np.random.randint(0, max_time_width + 1)
            if t == 0:
                continue
            t0 = np.random.randint(0, max(1, num_frames - t))
            mel[:, t0:t0+t] = 0

        return mel.astype(np.float32)

    def _get_audio_path(self, row):
        return os.path.join(
            self.dataset_path,
            "audio",
            f"fold{row['fold']}",
            row["slice_file_name"]
        )

    def _get_cache_path(self, row):
        stem = os.path.splitext(row["slice_file_name"])[0]
        return os.path.join(
            self.cache_dir,
            f"{stem}_sr{self.sr}_mel{self.n_mels}_len{self.target_length}.npy"
        )

    def _is_valid_mel(self, mel_db):
        return (
            isinstance(mel_db, np.ndarray)
            and mel_db.size > 0
            and mel_db.shape == self.expected_shape
            and np.isfinite(mel_db).all()
        )

    def _atomic_save(self, array, cache_path):
        tmp_path = f"{cache_path}.tmp.{os.getpid()}.{uuid.uuid4().hex}.npy"
        np.save(tmp_path, array)
        os.replace(tmp_path, cache_path)

    def _build_and_cache_mel(self, file_path, cache_path):
        y = self.load_audio(file_path)
        mel_db = self.extract_mel_spectrogram(y)
        self._atomic_save(mel_db, cache_path)
        return mel_db

    def _load_or_create_cached_mel(self, file_path, cache_path):
        if os.path.exists(cache_path):
            try:
                mel_db = np.load(cache_path)
                if self._is_valid_mel(mel_db):
                    return mel_db
                raise ValueError(f"bad cache shape={getattr(mel_db, 'shape', None)}")
            except Exception as e:
                print(f"[Cache rebuild] {os.path.basename(cache_path)} -> {e}")
                try:
                    os.remove(cache_path)
                except OSError:
                    pass

        return self._build_and_cache_mel(file_path, cache_path)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        file_path = self._get_audio_path(row)

        if self.augment:
            y = self.load_audio(file_path)
            y = self.augment_audio(y)
            mel_db = self.extract_mel_spectrogram(y)
            mel_db = self.spec_augment(mel_db)
        else:
            cache_path = self._get_cache_path(row)
            mel_db = self._load_or_create_cached_mel(file_path, cache_path)

        mel_tensor = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0)
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.long)

        return mel_tensor, label_tensor

In [13]:
def get_fold_split(metadata, test_fold, val_fold=None):
    """
    Разбиение по official folds.

    Если val_fold задан:
        train = все fold'ы, кроме test_fold и val_fold
        val   = val_fold
        test  = test_fold

    Если val_fold не задан:
        train = все fold'ы, кроме test_fold
        test  = test_fold
    """
    test_df = metadata[metadata["fold"] == test_fold]

    if val_fold is not None:
        assert val_fold != test_fold
        val_df = metadata[metadata["fold"] == val_fold]
        train_folds = [f for f in range(1, 11) if f not in {test_fold, val_fold}]
    else:
        val_df = None
        train_folds = [f for f in range(1, 11) if f != test_fold]

    train_df = metadata[metadata["fold"].isin(train_folds)]

    print(
        f"test={test_fold}"
        + (f", val={val_fold}" if val_fold is not None else "")
        + f" | Train: {len(train_df)}, Val: {len(val_df) if val_df is not None else 0}, Test: {len(test_df)}"
    )
    return train_df, val_df, test_df

In [14]:
train_df, val_df, test_df = get_fold_split(metadata, test_fold=10, val_fold=9)

BATCH_SIZE = 64

train_dataset = UrbanSoundDataset(
    train_df,
    DATASET_PATH,
    global_label_encoder,
    augment=True,
)

val_dataset = UrbanSoundDataset(
    val_df,
    DATASET_PATH,
    global_label_encoder,
    augment=False,
)

test_dataset = UrbanSoundDataset(
    test_df,
    DATASET_PATH,
    global_label_encoder,
    augment=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

sample_batch, sample_labels = next(iter(train_loader))
print(f"Batch shape: {sample_batch.shape}")
print(f"Labels shape: {sample_labels.shape}")
print(f"First labels: {sample_labels[:5]}")

test=10, val=9 | Train: 7079, Val: 816, Test: 837
Batch shape: torch.Size([64, 1, 128, 173])
Labels shape: torch.Size([64])
First labels: tensor([7, 8, 7, 9, 3])


In [15]:
class UrbanSoundCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5, channels=[1, 32, 64, 128, 256], fc_dim=128):
        # TODO: Реализуйте инициализацию слоев вашей архитектуры
        super(UrbanSoundCNN, self).__init__()

        self.num_classes = num_classes
        self.dropout = dropout
        self.channels = channels
        self.fc_dim = fc_dim

        layers = []
        for i in range(len(channels) - 1):
            in_ch = channels[i]
            out_ch = channels[i + 1]

            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1))
            layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            layers.append(nn.Dropout2d(0.2))

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(fc_dim, num_classes)
        )

    def forward(self, x):
        # TODO: Реализуйте forward вашей архитектуры
        x = self.features(x)
        x = self.classifier(x)
        return x

In [16]:
# TODO: Реализуйте функцию train_epoch(model, loader, criterion, optimizer, device)
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

# TODO: Реализуйте функцию validate(model, loader, criterion, device)
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

In [17]:
model = UrbanSoundCNN(num_classes=train_dataset.num_classes).to(device)

# Считаем параметры
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

Total parameters: 422,986
Trainable parameters: 422,986

Model architecture:
UrbanSoundCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout2d(p=0.2, inplace=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU(inplace=True)
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Dropout2d(p=0.2, inplace=False)
    (10): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding

In [19]:
def evaluate_10fold(
    model_class,
    model_kwargs,
    train_fn,
    metadata,
    dataset_path,
    label_encoder,
    device="cuda",
    verbose=True,
):
    """
    10-fold CV по folds UrbanSound8K
    """
    accuracies = []

    for test_fold in range(1, 11):
        val_fold = 1 if test_fold == 10 else test_fold + 1
        train_folds = [f for f in range(1, 11) if f not in {test_fold, val_fold}]

        if verbose:
            print(f"\n{'=' * 50}")
            print(f"Fold {test_fold}/10 (val={val_fold}, train={train_folds})")

        train_df = metadata[metadata["fold"].isin(train_folds)]
        val_df = metadata[metadata["fold"] == val_fold]
        test_df = metadata[metadata["fold"] == test_fold]

        train_ds = UrbanSoundDataset(train_df, dataset_path, label_encoder, augment=True)
        val_ds = UrbanSoundDataset(val_df, dataset_path, label_encoder, augment=False)
        test_ds = UrbanSoundDataset(test_df, dataset_path, label_encoder, augment=False)

        loader_kwargs = dict(num_workers=0, pin_memory=False)
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, **loader_kwargs)
        val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, **loader_kwargs)
        test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, **loader_kwargs)
        model = model_class(**model_kwargs).to(device)
        model = train_fn(model, train_loader, val_loader, device)

        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                preds = model(inputs).argmax(1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())

        acc = np.mean(np.array(all_preds) == np.array(all_labels))
        accuracies.append(acc)

        if verbose:
            print(f"Fold {test_fold} accuracy: {acc * 100:.2f}%")

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)

    if verbose:
        print(f"\n{'=' * 50}")
        print(f"10-fold CV: {mean_acc * 100:.2f}% ± {std_acc * 100:.2f}%")

    return accuracies

In [20]:
import time
import numpy as np

def my_train_fn(model, train_loader, val_loader, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=1
    )

    NUM_EPOCHS = 15
    PATIENCE = 2

    best_val_acc = 0.0
    best_state = None
    patience_counter = 0
    epoch_times = []

    fold_start = time.time()

    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()
        print(f"  Starting epoch {epoch+1}/{NUM_EPOCHS}...")

        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        scheduler.step(val_loss)

        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)

        avg_epoch_time = np.mean(epoch_times)
        remaining_epochs = NUM_EPOCHS - (epoch + 1)
        est_remaining = avg_epoch_time * remaining_epochs

        print(
            f"  Epoch {epoch+1}/{NUM_EPOCHS} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.2f}% | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.2f}% | "
            f"time={epoch_time/60:.1f} min | "
            f"ETA fold={est_remaining/60:.1f} min"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    total_fold_time = time.time() - fold_start
    print(f"  Fold training time: {total_fold_time/60:.1f} min")

    return model

In [22]:
model_config = {
    "num_classes": 10,
    "channels": [1, 32, 64, 128, 256, 512],
    "fc_dim": 256,
}
def evaluate_10fold_pretrained(
    model_class,
    model_kwargs,
    train_fn,
    metadata,
    dataset_path,
    label_encoder,
    pretrained_path,
    device="cpu",
    verbose=True,
):
    accuracies = []

    for test_fold in range(1, 11):
        val_fold = 1 if test_fold == 10 else test_fold + 1
        train_folds = [f for f in range(1, 11) if f not in {test_fold, val_fold}]

        if verbose:
            print(f"\n{'=' * 50}")
            print(f"Fold {test_fold}/10 (val={val_fold}, train={train_folds})")

        train_df = metadata[metadata["fold"].isin(train_folds)]
        val_df = metadata[metadata["fold"] == val_fold]
        test_df = metadata[metadata["fold"] == test_fold]

        train_ds = UrbanSoundDataset(train_df, dataset_path, label_encoder, augment=True)
        val_ds = UrbanSoundDataset(val_df, dataset_path, label_encoder, augment=False)
        test_ds = UrbanSoundDataset(test_df, dataset_path, label_encoder, augment=False)

        loader_kwargs = dict(num_workers=2, pin_memory=True)
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, **loader_kwargs)
        val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, **loader_kwargs)
        test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False, **loader_kwargs)

        model = model_class(**model_kwargs).to(device)

        state_dict = torch.load(pretrained_path, map_location=device)
        model.load_state_dict(state_dict, strict=False)

        if verbose:
            print(f"Loaded pretrained weights from: {pretrained_path}")

        model = train_fn(model, train_loader, val_loader, device)

        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                preds = model(inputs).argmax(1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())

        acc = np.mean(np.array(all_preds) == np.array(all_labels))
        accuracies.append(acc)

        if verbose:
            print(f"Fold {test_fold} accuracy: {acc * 100:.2f}%")

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)

    if verbose:
        print(f"\n{'=' * 50}")
        print(f"10-fold CV: {mean_acc * 100:.2f}% ± {std_acc * 100:.2f}%")

    return accuracies

In [29]:
accuracies = evaluate_10fold_pretrained(
    model_class=UrbanSoundCNN,
    model_kwargs=model_config,
    train_fn=my_train_fn,
    metadata=metadata,
    dataset_path=DATASET_PATH,
    label_encoder=global_label_encoder,
    pretrained_path = "/kaggle/input/datasets/juli06/predtren/urbansound_pretrained.pth",
    device="cuda",
)


Fold 1/10 (val=2, train=[3, 4, 5, 6, 7, 8, 9, 10])
Loaded pretrained weights from: /kaggle/input/datasets/juli06/predtren/urbansound_pretrained.pth
  Starting epoch 1/15...
  Epoch 1/15 | train_loss=0.8361 train_acc=72.77% | val_loss=0.4881 val_acc=84.12% | time=3.8 min | ETA fold=53.3 min
  Starting epoch 2/15...
  Epoch 2/15 | train_loss=0.7857 train_acc=73.95% | val_loss=0.5463 val_acc=81.98% | time=3.3 min | ETA fold=46.3 min
  Starting epoch 3/15...
  Epoch 3/15 | train_loss=0.7588 train_acc=74.41% | val_loss=0.7296 val_acc=71.40% | time=3.3 min | ETA fold=41.7 min
  Early stopping at epoch 3
  Fold training time: 10.4 min
Fold 1 accuracy: 87.29%

Fold 2/10 (val=3, train=[1, 4, 5, 6, 7, 8, 9, 10])
Loaded pretrained weights from: /kaggle/input/datasets/juli06/predtren/urbansound_pretrained.pth
  Starting epoch 1/15...
  Epoch 1/15 | train_loss=0.7943 train_acc=74.07% | val_loss=0.6038 val_acc=78.59% | time=3.8 min | ETA fold=53.0 min
  Starting epoch 2/15...
  Epoch 2/15 | train_l

In [30]:
mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

for i, acc in enumerate(accuracies, 1):
    print(f"Fold {i}: {acc * 100:.2f}%")

print(f"\n10-fold CV: {mean_acc * 100:.2f}% ± {std_acc * 100:.2f}%")

Fold 1: 87.29%
Fold 2: 81.08%
Fold 3: 82.92%
Fold 4: 78.59%
Fold 5: 82.69%
Fold 6: 81.90%
Fold 7: 83.77%
Fold 8: 82.75%
Fold 9: 76.72%
Fold 10: 76.70%

10-fold CV: 81.44% ± 3.14%


Баллы за пункт 2.4 начисляются в зависимости от лучшего выполненного порога по средней accuracy в 10-fold CV: чем выше итоговое качество, тем больше доля из 2,5 балла за эту часть.

In [31]:
score_2_4 = 0.0

if mean_acc >= 0.64:
    score_2_4 = 0.5 * 2.5
if mean_acc >= 0.69:
    score_2_4 = 0.6 * 2.5
if mean_acc >= 0.74:
    score_2_4 = 0.7 * 2.5
if mean_acc >= 0.79:
    score_2_4 = 0.8 * 2.5
if mean_acc >= 0.84:
    score_2_4 = 0.9 * 2.5
if mean_acc >= 0.88:
    score_2_4 = 1.0 * 2.5

print(f"Mean CV accuracy: {mean_acc:.4f}")
print(f"Баллы за пункт 2.4: {score_2_4:.2f} / 2.5")

Mean CV accuracy: 0.8144
Баллы за пункт 2.4: 2.00 / 2.5


In [34]:
torch.save(model.state_dict(), "/kaggle/working/best_cnn_model.pth")